# Recipe RAG — AI Diet Coach

A RAG pipeline over TheMealDB recipe data, following the course pattern:
`search` → `build_prompt` → `llm` → `rag`.

The index covers recipe name, category, ingredients, and instructions so the
coach can answer questions about diet goals, cooking time, and meal types.

In [1]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load Recipe Data

103 recipes fetched from TheMealDB across 14 categories. Each record has
`name`, `category`, `ingredients`, `instructions`, and `cooking_time_minutes`.

In [2]:
with open('../data/recipes.json') as f:
    documents = json.load(f)

print(f'Loaded {len(documents)} recipes')
print('Sample:', documents[0]['name'], '|', documents[0]['category'],
      '|', documents[0]['cooking_time_minutes'], 'min')

Loaded 103 recipes
Sample: Algerian Kefta (Meatballs) | Beef | 20 min


## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [3]:
index = Index(
    text_fields=["name", "category", "ingredients", "instructions"],
    keyword_fields=["category"],
)

index.fit(documents)
print('Index built on', len(documents), 'recipes')

Index built on 103 recipes


## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [4]:
def search(query):
    return index.search(query, num_results=5)

In [5]:
instructions = """
You are an AI diet coach helping users reach their weight-loss goals.
Recommend recipes from the CONTEXT that match the user's QUESTION.
For each recommendation include: recipe name, category, key ingredients,
estimated cooking time, and a one-sentence reason why it fits the request.
If the context contains no suitable match, say so and suggest a general tip.
Never invent recipes not present in the context.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [6]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [7]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

In [8]:
print(rag("I want a high protein meal I can cook in under 30 minutes"))

Here are some high protein meal recommendations you can cook in under 30 minutes:

1. **Beef and Broccoli Stir-Fry**  
   - **Category:** Beef  
   - **Key Ingredients:** Sirloin steak, Broccoli, Soy Sauce, Garlic  
   - **Estimated Cooking Time:** 14 minutes  
   - **Reason:** This dish is quick to prepare and packed with protein from the beef, making it ideal for your request.

2. **Algerian Kefta (Meatballs)**  
   - **Category:** Beef  
   - **Key Ingredients:** Ground Beef, Garlic, Onion, Tomatoes  
   - **Estimated Cooking Time:** 20 minutes  
   - **Reason:** These flavorful meatballs are high in protein and can be made swiftly, fitting your time constraints perfectly.

These recipes will help you meet your high protein meal needs in under half an hour!
